# Train the cosmology-dependent Velocileptors PT-coefficient emulator

This notebook trains a neural network to map the eight independent cosmological parameters to the native Velocileptors PT component table. The network target is the full table, shape (3, 117, 19), represented through component scaling and PCA. Nuisance parameters and the final output k sampling are applied only after the network prediction.

**Development dataset:** velocileptors_pt_lhs_v1 (248 available cosmologies). This run is known to be incomplete/unsuitable as a final scientific training set. It is used here to exercise file loading, preprocessing, NN fitting, coefficient contraction, and the final theory-comparison workflow. Switch RUN_NAME to velocileptors_pt_lhs_v2_restricted when all 300 tables are ready.

The emulator-training section does not call CLASS or evaluate theory at arbitrary cosmologies. The final, explicitly separated comparison section does so solely to obtain the cosmology-dependent growth rate needed by the analytic nuisance contraction and to calculate the reference spectrum.

Run from the project checkout in the configured NERSC Jupyter environment. The project package is imported as src; no sys.path modifications are made.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

np.set_printoptions(precision=6, suppress=True)
RANDOM_SEED = 20261112

## 1. Select the training run

Only change RUN_NAME when switching from the v1 plumbing test to the completed v2 dataset. The notebook reads one compressed file per cosmology from pt_tables/.

In [ ]:
DATA_ROOT = Path("./training_data")
RUN_NAME = "velocileptors_pt_lhs_v1"  # change to velocileptors_pt_lhs_v2_restricted when ready
RUN_DIR = DATA_ROOT / RUN_NAME
TABLE_DIR = RUN_DIR / "pt_tables"

if not TABLE_DIR.is_dir():
    raise FileNotFoundError(f"PT-table directory not found: {TABLE_DIR.resolve()}")

with open(RUN_DIR / "run_config.json") as stream:
    run_config = json.load(stream)

COSMOLOGY_PARAMETERS = [
    "omega_b", "omega_cdm", "sigma8_cb", "n_s",
    "alpha_s", "N_ur", "w0_fld", "wa_fld",
]
EXPECTED_ELL = np.array([0, 2, 4], dtype=int)
EXPECTED_TABLE_SHAPE = (3, 117, 19)

print("Run:", RUN_NAME)
print("Configured cosmologies:", run_config.get("n_cosmologies"))
print("Table directory:", TABLE_DIR.resolve())

## 2. Load and validate the native PT tables

Each NPZ file is expected to contain pt_table, k_native, ells, component_labels, cosmology_json, and metadata_json. We retain only the eight independent cosmological coordinates as network inputs. Component labels are generic indices; the network does not assume a physical interpretation for an individual component.

In [ ]:
table_files = sorted(TABLE_DIR.glob("cosmo_*.npz"))
if not table_files:
    raise FileNotFoundError(f"No cosmo_*.npz files found in {TABLE_DIR}")

records, tables = [], []
native_k_reference = None
component_labels_reference = None

for path in table_files:
    with np.load(path, allow_pickle=False) as saved:
        required = {"pt_table", "k_native", "ells", "component_labels",
                    "cosmology_json", "metadata_json"}
        missing = required.difference(saved.files)
        if missing:
            raise KeyError(f"{path.name} is missing keys: {sorted(missing)}")
        table = np.asarray(saved["pt_table"], dtype=np.float64)
        k_native = np.asarray(saved["k_native"], dtype=np.float64)
        ells = np.asarray(saved["ells"], dtype=int)
        labels = saved["component_labels"].astype(str).tolist()
        cosmo = json.loads(str(saved["cosmology_json"].item()))
        metadata = json.loads(str(saved["metadata_json"].item()))

    if table.shape != EXPECTED_TABLE_SHAPE:
        raise ValueError(f"{path.name}: table shape {table.shape}, expected {EXPECTED_TABLE_SHAPE}")
    if not np.isfinite(table).all():
        raise ValueError(f"{path.name}: non-finite PT-table values")
    if k_native.shape != (EXPECTED_TABLE_SHAPE[1],) or not np.isfinite(k_native).all():
        raise ValueError(f"{path.name}: invalid native k coordinate")
    if not np.all(np.diff(k_native) > 0):
        raise ValueError(f"{path.name}: native k must be strictly increasing")
    if not np.array_equal(ells, EXPECTED_ELL):
        raise ValueError(f"{path.name}: unexpected multipoles {ells}")
    if len(labels) != EXPECTED_TABLE_SHAPE[2]:
        raise ValueError(f"{path.name}: expected 19 component labels")
    if native_k_reference is None:
        native_k_reference, component_labels_reference = k_native.copy(), labels
    elif not np.allclose(k_native, native_k_reference, rtol=0, atol=1e-12):
        raise ValueError(f"{path.name}: native k grid differs from the first file")
    elif labels != component_labels_reference:
        raise ValueError(f"{path.name}: component-label ordering changed")

    if not all(name in cosmo for name in COSMOLOGY_PARAMETERS):
        raise ValueError(f"{path.name}: missing cosmological coordinates")
    xrow = [float(cosmo[name]) for name in COSMOLOGY_PARAMETERS]
    if not np.isfinite(xrow).all():
        raise ValueError(f"{path.name}: non-finite cosmological coordinates")

    file_index = int(path.stem.split("_")[-1])
    if int(metadata.get("cosmology_index", -1)) != file_index:
        raise ValueError(f"{path.name}: filename and metadata indices disagree")
    records.append({"cosmology_index": file_index, **dict(zip(COSMOLOGY_PARAMETERS, xrow))})
    tables.append(table)

X_all = pd.DataFrame(records).sort_values("cosmology_index").reset_index(drop=True)
order = np.argsort([r["cosmology_index"] for r in records])
Y_table_all = np.stack(tables, axis=0)[order]
if X_all["cosmology_index"].duplicated().any():
    raise ValueError("Duplicate cosmology indices found")

X = X_all[COSMOLOGY_PARAMETERS].to_numpy(dtype=np.float64)
n_cosmo = len(X)
print(f"Loaded {n_cosmo} cosmologies")
print("X shape:", X.shape)
print("PT table array shape:", Y_table_all.shape)
print("Native k range:", (native_k_reference.min(), native_k_reference.max()))
print("Multipoles:", EXPECTED_ELL)

## 3. Prepare the coefficient targets

The raw target has shape (N_cosmology, 3, 117, 19). We scale each component by its RMS over the training cosmologies, multipoles, and native k samples, then flatten each table and use PCA to obtain compact target coordinates for the neural network.

All scalers and PCA transformations are fitted on the training partition only. Holdout cosmologies are not used in preprocessing or model fitting.

In [ ]:
X_train_raw, X_valid_raw, Y_train_table, Y_valid_table = train_test_split(
    X, Y_table_all, test_size=0.20, random_state=RANDOM_SEED
)
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train_raw)
X_valid = x_scaler.transform(X_valid_raw)

component_scale = np.sqrt(np.mean(Y_train_table**2, axis=(0, 1, 2)))
component_scale = np.maximum(component_scale, 1e-30)

def flatten_scaled_tables(table_array):
    return (table_array / component_scale[None, None, None, :]).reshape(len(table_array), -1)

Y_train_flat = flatten_scaled_tables(Y_train_table)
Y_valid_flat = flatten_scaled_tables(Y_valid_table)
MAX_PCA_COMPONENTS = min(48, len(X_train) - 1, Y_train_flat.shape[1])
pca = PCA(n_components=MAX_PCA_COMPONENTS, svd_solver="full")
Z_train = pca.fit_transform(Y_train_flat)
Z_valid = pca.transform(Y_valid_flat)

print("Training cosmologies:", len(X_train))
print("Holdout cosmologies:", len(X_valid))
print("PCA dimensions:", pca.n_components_)
print("Cumulative PCA variance:", np.cumsum(pca.explained_variance_ratio_)[-1])

## 4. Train the neural network

The MLP maps the standardized eight-dimensional cosmology vector to the PCA coordinates of the PT table. Held-out cosmologies provide a basic end-to-end generalization check. Since v1 is a failed/incomplete design, these scores are software diagnostics, not a performance claim for the final training design.

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(
    hidden_layer_sizes=(128, 128), activation="tanh", solver="adam",
    alpha=1e-4, learning_rate_init=3e-4, max_iter=3000,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=100, tol=1e-6,
    random_state=RANDOM_SEED, verbose=True,
)
fit_start = time.perf_counter()
mlp.fit(X_train, Z_train)
fit_seconds = time.perf_counter() - fit_start

Z_valid_pred = mlp.predict(X_valid)
Y_valid_flat_pred = pca.inverse_transform(Z_valid_pred)
Y_valid_pred = Y_valid_flat_pred.reshape(Y_valid_table.shape) * component_scale[None, None, None, :]

def table_error_summary(truth, prediction):
    diff = prediction - truth
    denom = max(float(np.sqrt(np.mean(truth**2))), 1e-30)
    return {
        "relative_rms": float(np.sqrt(np.mean(diff**2)) / denom),
        "max_abs": float(np.max(np.abs(diff))),
        "mean_abs": float(np.mean(np.abs(diff))),
    }

print(f"Fit time: {fit_seconds:.1f} s")
print("Holdout PT-table errors:", table_error_summary(Y_valid_table, Y_valid_pred))

## 5. Inspect training diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mlp.loss_curve_)
ax.set_xlabel("Iteration")
ax.set_ylabel("MLP training loss")
ax.set_title("Neural-network optimization")
ax.grid(alpha=0.25)
plt.show()

per_cosmo_error = np.sqrt(np.mean((Y_valid_pred - Y_valid_table)**2, axis=(1, 2, 3)))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(len(per_cosmo_error)), per_cosmo_error, "o")
ax.set_xlabel("Holdout cosmology")
ax.set_ylabel("PT-table RMSE")
ax.set_title("Held-out cosmology errors (v1 diagnostic)")
ax.grid(alpha=0.25)
plt.show()

## 6. Refit the final emulator on every available cosmology

After recording the holdout diagnostic, refit preprocessing and the network using all available tables. This is the model used in the arbitrary-cosmology cell below.

In [ ]:
x_scaler_final = StandardScaler().fit(X)
X_all_scaled = x_scaler_final.transform(X)
component_scale_final = np.sqrt(np.mean(Y_table_all**2, axis=(0, 1, 2)))
component_scale_final = np.maximum(component_scale_final, 1e-30)
Y_all_flat = (Y_table_all / component_scale_final[None, None, None, :]).reshape(n_cosmo, -1)

pca_final = PCA(n_components=min(MAX_PCA_COMPONENTS, n_cosmo - 1), svd_solver="full")
Z_all = pca_final.fit_transform(Y_all_flat)

emulator = MLPRegressor(
    hidden_layer_sizes=(128, 128), activation="tanh", solver="adam",
    alpha=1e-4, learning_rate_init=3e-4, max_iter=3000,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=100, tol=1e-6,
    random_state=RANDOM_SEED, verbose=True,
)
emulator.fit(X_all_scaled, Z_all)
print("Final emulator trained on all", n_cosmo, "available cosmologies.")
print("Final PCA variance retained:", np.cumsum(pca_final.explained_variance_ratio_)[-1])

## 7. Define arbitrary-cosmology prediction and PT-table contraction helpers

The NN returns the predicted native table. The exact Velocileptors nuisance mapping converts physical nuisance parameters into the standard coefficient vector, and the installed 19-component contraction helper combines that vector with the table.

The growth rate f enters this analytic nuisance mapping. We obtain it from the initialized theory object in the final comparison section; it is not an NN target. This section defines prediction without evaluating theory.

In [ ]:
def predict_pt_table(cosmology):
    """Predict one native (ell, k_native, component) PT table."""
    missing = [name for name in COSMOLOGY_PARAMETERS if name not in cosmology]
    if missing:
        raise KeyError(f"Missing cosmological parameters: {missing}")
    x = np.array([[float(cosmology[name]) for name in COSMOLOGY_PARAMETERS]])
    if not np.isfinite(x).all():
        raise ValueError("Cosmological inputs must be finite.")
    z = emulator.predict(x_scaler_final.transform(x))
    flat = pca_final.inverse_transform(z)
    table = flat.reshape(1, 3, len(native_k_reference), 19)
    table *= component_scale_final[None, None, None, :]
    return table[0]


def reconstruct_spectrum_from_table(pt_table, growth_rate, nuisance, stochastic_settings, nbar):
    """Contract a predicted native PT table and interpolate to the final output k grid."""
    from desilike.theories.galaxy_clustering.full_shape import (
        _velocileptors_physical_to_standard,
        _velocileptors_combine_bias_terms_spectrum2_poles,
    )
    standard = _velocileptors_physical_to_standard(
        b1=nuisance["b1"], b2=nuisance["b2"], bs=nuisance["bs"], b3=nuisance["b3"],
        alpha0=nuisance["alpha0"], alpha2=nuisance["alpha2"],
        alpha4=nuisance["alpha4"], alpha6=nuisance["alpha6"],
        sn0=nuisance["sn0"], sn2=nuisance["sn2"], sn4=nuisance["sn4"],
        f=float(growth_rate), fsat=float(stochastic_settings["fsat"]),
        sigv=float(stochastic_settings["sigv"]), nbar=float(nbar),
        A=1.0, A_AP=1.0, rept=False,
    )
    native_pk = np.asarray(
        _velocileptors_combine_bias_terms_spectrum2_poles(pt_table, np.asarray(standard, dtype=float)),
        dtype=float,
    )
    if native_pk.shape != (3, len(native_k_reference)):
        raise ValueError(f"Unexpected reconstructed shape: {native_pk.shape}")
    output_pk = np.vstack([np.interp(K_FID, native_k_reference, row) for row in native_pk])
    return output_pk, native_pk

## 8. Arbitrary cosmology: emulator versus true theory

Edit the eight cosmological coordinates below. Keep the point inside the training design's rectangular domain; extrapolation is not validated. Nuisance parameters are specified separately and applied only during the analytic contraction. The default is the fiducial nuisance choice: b1=2 and all other nuisance parameters zero.

This final section is the only place where the notebook initializes/evaluates external theory. The direct theory spectrum is the reference; the emulator spectrum is reconstructed from the NN-predicted PT table.

In [ ]:
TEST_COSMOLOGY = {
    "omega_b": 0.02237, "omega_cdm": 0.12000, "sigma8_cb": 0.714197,
    "n_s": 0.9649, "alpha_s": 0.0, "N_ur": 2.0328,
    "w0_fld": -1.0, "wa_fld": 0.0,
}
FIDUCIAL_NUISANCE = {
    "b1": 2.0, "b2": 0.0, "bs": 0.0, "b3": 0.0,
    "alpha0": 0.0, "alpha2": 0.0, "alpha4": 0.0, "alpha6": 0.0,
    "sn0": 0.0, "sn2": 0.0, "sn4": 0.0,
}

from src.theory import TheoryManager, Z_FID, NBAR_FID, K_FID, ELL_FID
from desilike.theories.galaxy_clustering.full_shape import get_physical_stochastic_settings

if not np.array_equal(np.asarray(ELL_FID, dtype=int), EXPECTED_ELL):
    raise ValueError(f"Unexpected theory multipoles: {ELL_FID}")

theory_manager = TheoryManager(max_cache_size=1, z=Z_FID)
nuisance_for_theory = {
    "bias": {k: FIDUCIAL_NUISANCE[k] for k in ("b1", "b2", "bs", "b3")},
    "eft": {k: FIDUCIAL_NUISANCE[k] for k in ("alpha0", "alpha2", "alpha4", "alpha6")},
    "stochastic": {k: FIDUCIAL_NUISANCE[k] for k in ("sn0", "sn2", "sn4")},
}
pk_true = theory_manager.evaluate(TEST_COSMOLOGY, **nuisance_for_theory)
theory, _pipeline = theory_manager.get_theory(TEST_COSMOLOGY)

growth_rate = None
for obj in (getattr(theory, "template", None), theory, getattr(theory, "pt", None)):
    if obj is None:
        continue
    for attr in ("f", "growth_rate"):
        if hasattr(obj, attr):
            value = getattr(obj, attr)
            if np.isscalar(value):
                growth_rate = float(value)
                break
    if growth_rate is not None:
        break
if growth_rate is None:
    raise RuntimeError("Could not locate growth rate on initialized theory/template.")

stochastic_settings = get_physical_stochastic_settings("LRG")
pt_table_pred = predict_pt_table(TEST_COSMOLOGY)
pk_emulator, pk_emulator_native = reconstruct_spectrum_from_table(
    pt_table_pred, growth_rate, FIDUCIAL_NUISANCE, stochastic_settings, NBAR_FID
)

k_output = np.asarray(K_FID, dtype=float)
if pk_true.shape != pk_emulator.shape or pk_true.shape != (3, len(k_output)):
    raise ValueError(f"Unexpected output shapes: true={pk_true.shape}, emulator={pk_emulator.shape}")
residual = pk_emulator - pk_true
relative_rms = np.sqrt(np.mean(residual**2)) / max(np.sqrt(np.mean(pk_true**2)), 1e-30)
max_fractional = np.max(np.abs(residual) / np.maximum(np.abs(pk_true), 1e-12))
print("Growth rate:", growth_rate)
print("Predicted PT table:", pt_table_pred.shape)
print(f"Relative RMS spectrum residual: {relative_rms:.6e}")
print(f"Maximum pointwise fractional residual: {max_fractional:.6e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for i, (ell, ax) in enumerate(zip(ELL_FID, axes)):
    ax.plot(k_output, pk_true[i], label="true theory", lw=2)
    ax.plot(k_output, pk_emulator[i], "--", label="NN PT-table emulator", lw=1.8)
    ax.set_title(f"ell={ell}")
    ax.set_xlabel("k [h/Mpc]")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_ylabel("P_ell(k) [(h^-1 Mpc)^3]")
fig.suptitle("True theory vs. PT-coefficient emulator")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharex=True)
for i, (ell, ax) in enumerate(zip(ELL_FID, axes)):
    ax.axhline(0.0, color="black", lw=0.8)
    ax.plot(k_output, residual[i])
    ax.set_title(f"Residual, ell={ell}")
    ax.set_xlabel("k [h/Mpc]")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("P_emulator - P_true")
fig.tight_layout()
plt.show()

## 9. Handoff

For the v1 test, success means all available files load, table axes and native k are consistent, the NN fits without shape/API errors, and the final reconstruction cell runs. A poor emulator-versus-theory residual is possible with v1 and is not by itself a software failure.

When v2 is complete, set RUN_NAME to velocileptors_pt_lhs_v2_restricted and rerun from the top. The v2 design is the intended training set for meaningful diagnostics.

In [ ]:
print("Run:", RUN_NAME)
print("Training cosmologies:", n_cosmo)
print("Input dimension:", X.shape[1])
print("PCA output dimension:", pca_final.n_components_)
print("Native PT table shape:", Y_table_all.shape[1:])
print("Test cosmology:", TEST_COSMOLOGY)
print("Theory cache:", theory_manager.cache_info())